# 從官方棋譜重新產生中間檔（分段說明）：競賽一 Player（v7x）

這本筆記本把管線從官方棋譜一路列到最後一步，也就是**沒有主辦方提供的中間檔時，怎麼用官方資料＋我們的權重＋外部模型重新產生它們**。**每一段都標了硬體需求、原作者實測耗時，以及 Colab 免費版能不能跑完。預設所有重段落都不執行**（`RUN_* = False`）；請依需要打開。

> 先講結論：整條重跑在免費版 Colab 上**做不完**。最上游的整條重跑至少約需 76 小時 GPU（已知分項相加，缺的分項沒有紀錄）、認人網路 GPU 小時數十、HumanSL 掃描約 100 萬盤。可以在免費版完成的是：外部權重下載、切分與（小規模）特徵、測試端的抽取、最後一步組合（`reproduce_final.ipynb`，需要中間檔）。
> 官方資料（主辦方提供）不附；自訓權重在 GitHub Release；外部模型（KataGo、HumanSL、SE）不散布，只給官方下載網址。

| 段 | 內容 | 硬體 | 原作者實測耗時（平台） | Colab 免費版（T4、單次 ≤12 小時、約 12 GB RAM） |
|---|---|---|---|---|
| 0 | 安裝、外部權重（約 1.1 GB） | CPU | 約 15 分鐘 | 可 |
| 1 | 切分＋L0 手工特徵快取（100 萬盤） | CPU | 數十分鐘（14 核）；免費版只有 2 核 | **可能跑不完**（核心少、RAM 吃緊；可縮小規模試） |
| 2 | KataGo HumanSL 掃描（訓練端約 100 萬盤，1 visit） | GPU（KataGo CUDA） | 約 2.35 盤／秒（Colab L4）：第 1 層 43.8 units；第 2 層約 80 萬盤在學校 RTX8000 | **不可**（約 118 小時）；測試端（2–3 萬盤，約 4 小時）可 |
| 3 | Strength Estimator 抽取（只有舊題 v6 用） | GPU | 訓練端 685,231 盤 12.9 小時（RTX8000）；舊題測試端 90,000 盤約 1 小時 | 訓練端**不可**；測試端可 |
| 4 | KataGo 主幹向量、逐局面編碼器 pe86 | GPU | 主幹向量約 11 小時（兩 GPU 並行）；pe86 全局面抽取 9.2 小時（L4）、編碼器每個 12.6 分鐘 | **不可** |
| 5 | S 認人網路 finh_full／h0／h1 | GPU | 合計約 42 GPU 小時（Colab L4 20.8＋14.4、本機 6.2） | **不可** |
| 6 | Player booster＋X_e112／X_lr／X_p1r | CPU＋GPU | booster 約 13 分鐘；X_p1r 每網路約 8 分鐘（GPU） | 可（需前段輸出） |
| 7 | 舊題 800 列（v6） | GPU＋CPU | 測試端 SE 約 1 小時、HSL 約 4 小時 | 測試端可；**最後一步缺檔，見 `PIPELINE.md`** |
| 8 | 最後一步 | CPU | 約 1 分鐘 | 可 → `reproduce_final.ipynb` |

各段的輸入輸出在 `PIPELINE.md` 的步驟表；GPU 結果不會位元重現（`index_add_` 順序不固定、fp16、各平台 torch 版本不同；KataGo 掃描為 1 visit＋隨機對稱，重掃逐盤有雜訊）。CPU 的最後一步則位元重現。
**所以自己重算出來的中間檔，位元組不會和主辦方拿到的那份相同**；要位元組重現最終提交檔，需要主辦方提供的那一份中間檔（見 README）。

In [ ]:
# 設定。標示「待補」的值要等 GitHub repo 與 Release 建好後填入。
import os, sys, subprocess
from pathlib import Path

REPO_URL    = os.environ.get("REPO_URL", "待補")        # 例：https://github.com/<帳號>/aicup2026-go-player.git
RELEASE_URL = os.environ.get("RELEASE_URL", "待補")     # 我們自訓權重的下載網址，例：https://github.com/<帳號>/aicup2026-go-player/releases/download/<tag>
REPO_DIR    = Path(os.environ.get("REPO_DIR", "/content/aicup2026-go-player"))
# （環境變數只是方便本機測試；在 Colab 直接改上面的字串即可。）

def run(cmd, cwd=None, env=None, check=True):
    """執行指令並印出輸出（stdout＋stderr）。"""
    e = dict(os.environ, **(env or {}))
    r = subprocess.run(cmd, cwd=str(cwd or REPO_DIR), env=e, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(r.stdout[-6000:] if len(r.stdout) > 6000 else r.stdout)
    if check and r.returncode:
        raise RuntimeError(f"指令失敗（exit {r.returncode}）：{cmd}")
    return r


In [ ]:
# 1) 取得程式碼
if not (REPO_DIR / "colab").exists():
    assert REPO_URL != "待補", "請先把上一格的 REPO_URL 填成實際網址"
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], cwd="/")
print("repo:", REPO_DIR, "| commit:", run(["git", "rev-parse", "--short", "HEAD"], check=False).stdout.strip())


In [ ]:
# 2) 安裝（CPU 套件鎖版本；GPU 段落另外需要 torch，Colab 內建的即可，原作者版本見 requirements-gpu.txt）
run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
try:
    import torch; print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
except ImportError:
    print("沒有 torch（只做 CPU 段落可忽略）")
run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=False)

In [ ]:
# 3) 外部預訓練權重（來源網址、版本、授權見 README「外部資源」；下載後驗 sha256；不散布，只從官方網址取得）
sys.path.insert(0, str(REPO_DIR / "colab"))
import colab_utils as U
RUN_EXTERNAL = False        # True：下載約 1.1 GB（KataGo 執行檔與兩個網路、HumanSL .ckpt、SE go.tar.gz，並解出 se_go.pt）
if RUN_EXTERNAL:
    U.fetch_external(REPO_DIR)
# HumanSL .ckpt 在 torch 2.12 載入前，需用 KataGo 原始碼的 python/katago/train/migrate_defaultdict_to_dict.py 轉換
# （v1.15.0 的 checkpoint 內含 defaultdict）；抽主幹向量時固定用 profile rank_1d、日本規則。細節見各腳本頂端說明。

def show_doc(script, n=1800):
    """印出腳本頂端的說明（參數與輸入輸出以它為準）。"""
    s = (REPO_DIR / script).read_text(encoding="utf-8")
    i = s.find('"""'); j = s.find('"""', i + 3)
    print(f"--- {script} ---\n" + (s[i + 3:j] if i >= 0 and j > i else s[:n])[:n])

## 官方資料放置

主辦方資料不附在 repo。請依 README「官方資料放置」放到 `<repo>/data/`：

```
data/training/                 train_<rank>.csv …（1,000,000 盤）
data/contest_1104/             原測試集（Rank、Player 舊題）
data/contest_1111/             11/11 新增的 Player 測試集
```
Colab 可掛載 Google Drive 後複製或建符號連結；**資料不要放進 GitHub**。

In [ ]:
# 資料就位檢查（只檢查資料夾是否存在）
for d in ("data/training", "data/contest_1104", "data/contest_1111"):
    print(f"{d:22s}", "有" if (REPO_DIR / d).exists() else "沒有")
# from google.colab import drive; drive.mount("/content/drive")      # 若資料在 Drive：掛載後 ln -s 到 data/

## 段 1：資料處理（CPU）

切分一律以 player_id 分組（同一玩家不同時出現在訓練與評估）；L0 手工特徵快取是第一階段的基礎。
輸出：`data/splits/`、`data/cache/l0/`、`data/cache/games_meta.parquet`。

In [ ]:
RUN_DATA = False
ENV = {"PYTHONPATH": "src:scripts:."}
if RUN_DATA:
    run([sys.executable, "scripts/make_split.py"], env=ENV)
    run([sys.executable, "scripts/build_l0_cache.py", "--jobs", "2"], env=ENV)     # 免費版只有 2 核；全量 100 萬盤很可能跑不完
else:
    show_doc("scripts/make_split.py"); show_doc("scripts/build_l0_cache.py")

## 段 2：KataGo HumanSL 掃描（GPU）

固定設定在 `scripts/katago/scan_pins.json` 與 `analysis.cfg`（1 visit、11 個段位 profile、日本規則、前 60 個己方手）。`setup_katago.sh` 在 Colab 安裝 KataGo（CUDA 12.8）與兩個網路（驗證版本）。
掃描由 `scan_runner.py` 以分片（shard）執行、可續跑；訓練端全量約 100 萬盤，免費版做不完。測試端由 `predict_hsl.py` 準備分片（Player 兩個池的 2–3 萬盤）。

In [ ]:
RUN_KATAGO_SETUP = False       # 安裝 KataGo 與網路到 /content/katago（需 GPU 執行階段）
RUN_HSL_TEST_PREPARE = False   # 以測試檔準備測試端掃描分片到 data/test_scan（需官方資料）
if RUN_KATAGO_SETUP:
    run(["bash", "scripts/katago/setup_katago.sh"])
if RUN_HSL_TEST_PREPARE:
    run([sys.executable, "scripts/predict_hsl.py", "prepare", "--data-dir", "data/contest_1104", "--root", "data/test_scan"], env={"PYTHONPATH": "src:scripts:.", "GOGAME_COLOR_SEMANTICS": "actual"})
    # 之後逐片執行（可續跑；示意，--shards 依 data/test_scan/shards.parquet 的 id）：
    # run([sys.executable, "scripts/katago/scan_runner.py", "--shards", "S9001", "--sgf-dir", "data/test_scan/sgf", "--shards-table", "data/test_scan/shards.parquet",
    #      "--katago-dir", "/content/katago", "--out", "data/test_scan/out", "--work", "data/test_scan/work", "--platform", "colab"])
    # 入帳與組特徵表：predict_hsl.py ingest DIR → predict_hsl.py features
show_doc("scripts/predict_hsl.py", 900); show_doc("scripts/katago/scan_runner.py", 900)

## 段 3：Strength Estimator 抽取（GPU）

每盤取 32 個己方局面，`score` 頭平均與標準差（fp16、rotation 0）。需要 `data/external/se/extracted/se_go.pt`（見上方外部權重；sha256 `95fc2951…`）。SE 權重由中研院公開頁面下載，我們不再散布。
訓練端（685,231 盤，`scripts/experiments/e21/extract.py`）約 12.9 GPU 小時，免費版做不完；測試端用 `scripts/predict_se.py extract`。

In [ ]:
RUN_SE_SMOKE = False     # 對 Rank 測試檔抽 SE 的小型試跑（--limit 200）；需 GPU、se_go.pt 與官方資料
if RUN_SE_SMOKE:
    run([sys.executable, "scripts/predict_se.py", "extract", "--data-dir", "data/contest_1104", "--out-dir", ".scratch/se_smoke",
         "--model", "data/external/se/extracted/se_go.pt", "--device", "cuda", "--half", "--limit", "200", "--color-semantics", "actual"], env={"PYTHONPATH": "src:scripts:."})
show_doc("scripts/predict_se.py", 1500); show_doc("scripts/experiments/e21/extract.py", 900)

## 段 4–5：主幹向量、pe86、S 認人網路（GPU，免費版做不完）

| 內容 | 腳本（說明在各檔頂端） |
|---|---|
| 主幹向量 | `scripts/experiments/e13/{encode,train}.py`、`scripts/oneoff/e52_kata_trunk2.py`、`e56_trunk.py`、`e62_trunk.py`、`e69_trunk.py` |
| 逐局面編碼器 pe86 | `scripts/oneoff/e77_enc.py`、`e80_*.py`、`e86_*.py`（含 Colab 版 `e86_colab_prep.py`、`e86_colab_enc.py`）。**pe86 編碼器權重沒有保存**（只剩它產出的每盤向量），要重新訓練 |
| S 認人網路 | `scripts/experiments/e89/{data_prep,train_s,fastboard,driver}.py`、`e90/{make_data_h,train_e90h,driver_e90h,assemble_pe_h}.py`（需 numba 0.64.0）。權重 `finh_full／h0／h1` 與大網路 `finh_full` 在 Release，**只做推論不用重訓** |

我們沒有在這本筆記本內提供未驗證的一鍵指令：這幾段的參數很多，請依各腳本頂端的說明與 `PIPELINE.md` 的步驟表執行。下面的格子只印出說明。

In [ ]:
for s in ("scripts/experiments/e89/train_s.py", "scripts/experiments/e90/train_e90h.py", "scripts/oneoff/e86_colab_enc.py"):
    show_doc(s, 1200)

## 段 6：Player 的 booster 與外掛通道

- booster：`scripts/experiments/e90/e90_final.py`（CPU 約 13 分鐘；權重 `v7q_S.*` 在 Release）→ `real_v7q_S_{pub,priv}.npz`。
- X_e112：`scripts/experiments/e112/`（三個 S 網路的逐局面向量 → 局面核密度比對）。
- X_lr：`scripts/experiments/e123/`（**需 scikit-learn 1.9.1**）。
- X_p1r：`scripts/experiments/e127/`（成對比對網路；權重 `p1r_finh_h0／h1` 在 Release）。

成對比對網路 X_p1r 的訓練指令（原紀錄；每網路約 8 分鐘，VRAM 約 7.2 GB，T4 15 GB 夠用；需要上游逐局面向量，這些向量的重算屬段 4–5）：

In [ ]:
RUN_P1R_TRAIN = False    # 需 GPU 與 .scratch/e127 下的逐局面向量（訓練端向量已刪，要先重算段 4–5）
if RUN_P1R_TRAIN:
    (REPO_DIR / ".scratch/e127/logs").mkdir(parents=True, exist_ok=True)
    run(["bash", "scripts/experiments/e127/job_train.sh", "p1r", "cross",
         "--steps 12000 --lr 1e-3 --lr_p 3e-4 --B 8 --M 24,58 --mon_every 1000 --ckpt_every 2000 --cand_sa 1 --qry_sa 1 --drop 0.1 --resid 1"])
show_doc("scripts/experiments/e127/e127_train.py", 1200)

## 段 7：舊題 800 列（Player v6）

v6 = E04v2 reranker（L0）＋ SE 與 HSL 的馬氏距離融合（e′ = e − 0.5·λ_se·D_se − 0.5·λ_hsl·D_hsl；λ_se=0.5、λ_hsl=0.25）＋轉導解碼（τ=.9、3 輪、再 Hungarian）。執行順序與缺件見 `PIPELINE.md`「舊題 800 列」。下面是**測試端**的步驟（需官方資料與 GPU）；最後的 `fuse` 需要訓練端算出的度量檔 `metric_se.npz`、`metric_hsl.npz`，這兩個檔已不在原專案，**目前無法重跑到最後**。

先確認基底 v3 可由官方資料與 E04v2 模型重現（CPU，約 1.5 分鐘；預期 sha256 前 12 碼 `b51ae1671927`）。E04v2 權重在 Release（`player_e04v2.pkl`）：

In [ ]:
RUN_V3 = False           # 需官方資料 data/contest_1104 與 data/models_final/player_e04v2.pkl（Release 的 player_e04v2.pkl，或 scripts/predict.py train --task player --include-holdout 重訓）
RUN_V6_TEST_SIDE = False # 以下需 GPU：SE 90,000 盤與 HSL 26,000 盤
ENV6 = {"PYTHONPATH": "src:scripts:.", "GOGAME_COLOR_SEMANTICS": "actual"}
if RUN_V3:
    run([sys.executable, "scripts/oneoff/e27_player_fusion.py", "v3"], env=ENV6)
if RUN_V6_TEST_SIDE:
    run([sys.executable, "scripts/oneoff/e27_player_fusion.py", "games"], env=ENV6)             # → .scratch/e27/se_games.parquet、hsl_games_in.parquet
    import pandas as pd
    df = pd.read_parquet(REPO_DIR / ".scratch/e27/se_games.parquet"); (REPO_DIR / ".scratch/e27/in").mkdir(exist_ok=True)
    for k in range(9):                                                                            # 9 個 10,000 盤分塊（e27_merge_se.py 預期 c0..c8）
        df.iloc[k * 10000:(k + 1) * 10000].reset_index(drop=True).to_parquet(REPO_DIR / f".scratch/e27/in/c{k}.parquet", index=False)
        run([sys.executable, "scripts/predict_se.py", "extract", "--games-parquet", f".scratch/e27/in/c{k}.parquet", "--out-dir", f".scratch/e27/se_out/c{k}",
             "--model", "data/external/se/extracted/se_go.pt", "--device", "cuda", "--half", "--workers", "2", "--threads", "2"], env=ENV6)
    run([sys.executable, "scripts/oneoff/e27_merge_se.py"], env=ENV6)                              # 驗 9 個 manifest → .scratch/e27/se/se_test_games.parquet
    # HSL：run([sys.executable, "scripts/oneoff/e27_hsl_prepare.py", "data/test_scan_e27player"]) → scan_runner（段 2）→ predict_hsl.py --root … features
    #       → 複製 hsl_test_games.parquet 到 .scratch/e27/hsl/ → e27_player_fusion.py fuse（需 metric_se.npz、metric_hsl.npz）

## 段 8：最後一步

見 `colab/reproduce_final.ipynb`（需要主辦方提供的中間檔；數秒內重現最終提交檔並比對 sha256）。若你自己重算了上游，把產出放到與 `ASSETS.md` 相同的路徑，再用同樣的指令：

```
python final/make_player_final.py --out out/player_v7x.csv
```
（GPU 上游不會位元重現，所以自己重算的結果 sha256 會與上傳檔不同；差異大小請看 `PIPELINE.md` 與 README 的「已知限制」。）